# Week 7 — Integration, History & OLAP
**Name:** Hashim Nadeem  
**Roll number:** SU92-BDSFM-F25-001  
**Section:** BS DS-7a  
**Course:** Data Warehousing & Business Intelligence

The uploaded brief is Week 7, confirmed by the user. This notebook answers T1–T8 and implements the pandas lab. The original Excel workbook is retained as a source template; all answers are in this notebook. Run all cells from the extracted ZIP folder. Outputs are rebuilt deterministically in `results/`; no Power BI session is claimed.

**Grain:** one consignment per shipping day. Preserve distinct IDs within a carrier; collapse repeated IDs within a source and verified Road/Courier duplicate filings only. Weight and freight cost are additive; cost/kg is a ratio of summed cost to summed weight.

In [1]:
from pathlib import Path
import sys
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    raise FileNotFoundError('Open this notebook from the extracted project folder containing data/.')
print('Input folder:', ROOT / 'data')

Input folder: /workspace/scratch/9088f49f6af7/work/data


## T1 — Architecture

| Item | Tier | Reason |
|---|---|---|
| ShaheenEx original JSON feed | Source | Carrier-owned evidence, including metadata |
| Grams to kilograms | Staging / ETL transform | Conforms the target contract without altering extraction |
| RejectLog_April | Storage | Persisted operational audit table; produced by ETL, retained for remediation |
| Appended 1,805 raw rows | Staging | Shaped, unchanged source values |
| Type 2 DimCustomer | Storage | Maintains effective-dated versions |
| Integrated FactShipment | Storage | Atomic keyed warehouse facts |
| Province × month matrix | Presentation | Business-facing arrangement of measures |
| Alias lookup | Staging / ETL reference | Conforms raw spellings before dimension lookup |
| JSON status_codes block | Source | Carrier-supplied code definitions, not warehouse-approved members |
| April cost/kg figure | BI Application | A measure evaluated in the page's filter context |

At the **source** tier register and retain the new JSON feed; preserve old feed contracts. At **staging** add flattening, unit conversion and scoped deduplication; preserve raw evidence. At **storage** add Courier, lanes, calendar dates and customer versions; preserve historical fact keys. At **presentation** expose Courier and the new month; preserve published definitions and March figures. BI pages need refreshed filters and measures with the same ratio-of-sums definition.

Extraction preserves grams. Converting there loses the original unit evidence and makes replay vulnerable to double conversion. The transformation has a source-specific, named conversion and always starts from immutable raw files.

## T2 — Source mapping and profiling

| Contract | Marine | Rail | Road | Courier JSON | Transformation |
|---|---|---|---|---|---|
| ShipmentID | Shipment_ID | consignment_no | trip_id | consignment | Trim, uppercase; restore RL- prefix |
| ShipDate | Ship_Date | date | trip_date | booked_at | Parse exact source format; local Pakistan day for JSON |
| OriginCity | Origin_Port | from_station | from_city | pickup.city | Trim/casefold; resolve approved aliases |
| DestCity | Dest_Port | to_station | to_city | dropoff.city | Same city conformance |
| WeightKG | Weight_KG | net_tons | weight_kg | weight_grams | Parse numeric; rail ×1,000; courier ÷1,000 |
| FreightCostPKR | Cost_PKR | freight_charges | cost_pkr | charge.amount | Numeric, retain legitimate negative credit notes |
| Status | Status | consignment_status | status | status_code | Trim/casefold; codes 1–3 mapped; others Unknown |
| CustomerID | Customer_Ref | customer_code | customer_id | customer.ref | Conform registry aliases to canonical business ID |
| SourceMode | generated Marine | generated Rail | generated Road | generated Courier | Source metadata; service does not define transport mode |

SourceMode is supplied by the adapter from source identity, not a physical column in any source. The JSON carrier metadata identifies the fourth carrier. The brief's claim that this column exists physically in one source is therefore interpreted as metadata availability.

IDs, dates and statuses all need conformance in every source. All raw values remain separately available for audit. The following executable pipeline implements named steps S01–S11 and exposes the raw staging and profile before publishing facts.

In [2]:
from pathlib import Path
import pandas as pd, numpy as np, json, hashlib
ROOT=Path.cwd()
DATA=ROOT/'data'; OUT=ROOT/'results'; OUT.mkdir(exist_ok=True)
def read(name): return pd.read_csv(DATA/name)
def norm(v): return '' if pd.isna(v) else str(v).strip().casefold()
def run_pipeline():
    # S01 Extract and shape without correcting source values.
    columns=['ShipmentID','ShipDate','OriginCity','DestCity','WeightKG','FreightCostPKR','Status','CustomerID']
    specs=[('marine_april.csv','Marine',['Shipment_ID','Ship_Date','Origin_Port','Dest_Port','Weight_KG','Cost_PKR','Status','Customer_Ref']),('rail_april.xlsx','Rail',['consignment_no','date','from_station','to_station','net_tons','freight_charges','consignment_status','customer_code']),('road_april.csv','Road',['trip_id','trip_date','from_city','to_city','weight_kg','cost_pkr','status','customer_id'])]
    pieces=[]; structural=[]
    for filename,mode,src in specs:
        raw=pd.read_excel(DATA/filename,header=2) if mode=='Rail' else pd.read_csv(DATA/filename,skip_blank_lines=False)
        digest=hashlib.sha256((DATA/filename).read_bytes()).hexdigest()
        keep=~raw.isna().all(axis=1)
        if mode=='Road': keep &= raw.trip_id.fillna('').str.strip().ne('TOTAL')
        for i,row in raw[~keep].iterrows(): structural.append({'SourceFile':filename,'Locator':f'line {i+2}','Reason':'Blank line' if row.isna().all() else 'TOTAL footer'})
        frame=raw.loc[keep,src].copy(); frame.columns=columns
        frame['SourceMode']=mode;frame['SourceFile']=filename
        frame['Locator']=[f'Consignments!row {i+4}' if mode=='Rail' else f'line {i+2}' for i in frame.index]
        frame['SourceSHA256']=digest
        frame['RawRecord']=[json.dumps(raw.loc[i].to_dict(),default=str) for i in frame.index]
        pieces.append(frame)
    filename='courier_april.json'; payload=json.loads((DATA/filename).read_text()); digest=hashlib.sha256((DATA/filename).read_bytes()).hexdigest(); rows=[]
    for i,r in enumerate(payload['consignments']):
        rows.append([r.get('consignment'),r.get('booked_at'),r.get('pickup',{}).get('city'),r.get('dropoff',{}).get('city'),r.get('weight_grams'),r.get('charge',{}).get('amount'),r.get('status_code'),r.get('customer',{}).get('ref'),'Courier',filename,f'$.consignments[{i}]',digest,json.dumps(r)])
    pieces.append(pd.DataFrame(rows,columns=columns+['SourceMode','SourceFile','Locator','SourceSHA256','RawRecord']))
    staging=pd.concat(pieces,ignore_index=True)
    # S02 Conform city/status/customer lookups; preserve raw staging separately.
    cities=read('DimCity(4).csv'); aliases=read('alias_lookup(2).csv'); base=read('DimCustomer.csv'); status=read('DimStatus(4).csv'); changes=read('customer_changes.csv')
    alias_map={norm(r.RawSpelling):r.CityKey for r in aliases.itertuples()}
    city_names=cities.set_index('CityKey').CityName.to_dict(); provinces=cities.set_index('CityName').Province.to_dict()
    status_map={norm(r.StatusName):r.StatusKey for r in status.itertuples() if r.StatusKey!=-1}
    known=set(base.CustomerID.dropna())
    name_to_id={}
    for r in base.dropna(subset=['CustomerID']).itertuples():
        for alias in str(r.RawAliases).split('|'):name_to_id[' '.join(norm(alias).split())]=r.CustomerID
    registry=read('customers_master(1).csv')
    customer_alias={r.CustomerID:name_to_id.get(' '.join(norm(r.CustomerName).split()),r.CustomerID) for r in registry.itertuples()}
    t=staging.copy()
    t['ShipmentID']=t.ShipmentID.map(lambda x:'' if pd.isna(x) else str(x).strip().upper())
    t.loc[t.SourceMode.eq('Rail'),'ShipmentID']='RL-'+t.loc[t.SourceMode.eq('Rail'),'ShipmentID'].str.replace(r'\.0$','',regex=True)
    t['CustomerID']=t.CustomerID.map(lambda x:'' if pd.isna(x) else str(x).strip().upper())
    t['CustomerID']=t.CustomerID.map(lambda v:customer_alias.get(v,v))
    # S03 Normalize units and dates in transform, including local courier day.
    def parse_date(r):
        try:
            fmt={'Marine':'%d/%m/%Y','Road':'%Y-%m-%d'}.get(r.SourceMode)
            d=pd.to_datetime(r.ShipDate,format=fmt,errors='raise')
            if pd.isna(d): return pd.NaT
            if d.tzinfo is not None: d=d.tz_convert('Asia/Karachi').tz_localize(None)
            return d.normalize()
        except (ValueError,TypeError):return pd.NaT
    t['ShipDate']=t.apply(parse_date,axis=1)
    t['WeightKG']=pd.to_numeric(t.WeightKG,errors='coerce')*t.SourceMode.map({'Marine':1,'Rail':1000,'Road':1,'Courier':.001})
    t['FreightCostPKR']=pd.to_numeric(t.FreightCostPKR,errors='coerce')
    for field,key in [('OriginCity','OriginCityKey'),('DestCity','DestCityKey')]:t[key]=t[field].map(lambda v:alias_map.get(norm(v),-1))
    t['StatusKey']=[({1:1,2:2,3:3}.get(int(v),-1) if pd.notna(v) else -1) if m=='Courier' else status_map.get(norm(v),-1) for v,m in zip(t.Status,t.SourceMode)]
    failures=pd.DataFrame(index=t.index)
    failures['ShipmentID']=t.ShipmentID.eq('')
    failures['ShipDate']=t.ShipDate.isna()|~t.ShipDate.between('2026-04-01','2026-04-30')
    failures['OriginCity']=t.OriginCityKey.eq(-1); failures['DestCity']=t.DestCityKey.eq(-1)
    failures['WeightKG']=t.WeightKG.isna()|t.WeightKG.le(0)|t.WeightKG.gt(t.SourceMode.map({'Marine':25000,'Rail':35000,'Road':8000,'Courier':8000}))
    failures['FreightCostPKR']=t.FreightCostPKR.isna();failures['Status']=t.StatusKey.eq(-1);failures['CustomerID']=~t.CustomerID.isin(known)
    # S04 Profile raw values against the conformed rules, without mutating staging.
    prof=[]
    for c in columns:
        raw=staging[c]; blank=raw.isna()|raw.astype(str).str.strip().eq('')
        normalized=t[c]
        text_only=raw.astype(str).ne(normalized.astype(str))&~failures[c]&~blank if c in ['ShipmentID','OriginCity','DestCity','Status','CustomerID'] else pd.Series(False,index=t.index)
        if c in ['OriginCity','DestCity']:text_only=raw.map(lambda v:str(v) != city_names.get(alias_map.get(norm(v),-1),str(v)))&~failures[c]&~blank
        if c=='Status':text_only=pd.Series([m!='Courier' and str(v)!=status.set_index('StatusKey').StatusName.get(k,'') for v,m,k in zip(raw,t.SourceMode,t.StatusKey)])&~failures[c]
        prof.append([c,int(blank.sum()),raw.nunique(dropna=True),int(failures[c].sum()),int(text_only.sum())])
    profile=pd.DataFrame(prof,columns=['Column','Blanks','DistinctRaw','RuleFailures','TextTransformOnly'])
    # S05 Reject invalid measures, dates or IDs. All failed reasons retained.
    reason_masks={'WEIGHT_NOT_NUMERIC':t.WeightKG.isna(),'WEIGHT_NONPOSITIVE':t.WeightKG.le(0),'WEIGHT_OVER_CAPACITY':t.WeightKG.gt(t.SourceMode.map({'Marine':25000,'Rail':35000,'Road':8000,'Courier':8000})),'MISSING_SHIPMENT_ID':failures.ShipmentID,'INVALID_APRIL_DATE':failures.ShipDate,'COST_NOT_NUMERIC':failures.FreightCostPKR}
    invalid=pd.DataFrame(reason_masks).any(axis=1)
    reject=t.loc[invalid,['SourceFile','SourceSHA256','Locator','ShipmentID','RawRecord']].copy()
    reject['ReasonCode']=[';'.join(k for k,v in reason_masks.items() if v.loc[i]) for i in reject.index]
    reject['LoadBatch']='2026-04'
    good=t.loc[~invalid].copy()
    # S06 Deduplicate only same source + normalized shipment ID.
    same=good.duplicated(['SourceFile','ShipmentID'],keep='first');dups=good.loc[same].copy();dups['RemovalReason']='Same source / ShipmentID';good=good.loc[~same].copy()
    # S07 Remove one courier record per matching road consignment, never road-road.
    bk=['ShipDate','OriginCityKey','DestCityKey','WeightKG','CustomerID']
    road=good.loc[good.SourceMode.eq('Road')].copy(); courier=good.loc[good.SourceMode.eq('Courier')].copy()
    road['Occurrence']=road.groupby(bk,dropna=False).cumcount();courier['Occurrence']=courier.groupby(bk,dropna=False).cumcount()
    matches=courier.reset_index().merge(road[bk+['Occurrence','ShipmentID']],on=bk+['Occurrence'],suffixes=('_Courier','_Road'))
    cross=good.loc[matches['index']].copy();cross['RemovalReason']='Road / Courier same consignment';dups=pd.concat([dups,cross]);good=good.drop(index=cross.index)
    # S08 Build deterministic Type 2 snapshot. Rebuild from immutable carryforward on each run.
    dim=base.copy();dim['ValidFrom']='1900-01-01';dim['ValidTo']='9999-12-31';dim['RowIsCurrent']='Yes'
    for change in changes.itertuples():
        ix=dim.index[(dim.CustomerID==change.CustomerID)&dim.RowIsCurrent.eq('Yes')][0]
        new=dim.loc[ix].copy();effective=pd.Timestamp(change.EffectiveDate)
        dim.loc[ix,['ValidTo','RowIsCurrent']]=[(effective-pd.Timedelta(days=1)).strftime('%Y-%m-%d'),'No']
        new['CustomerKey']=int(dim.CustomerKey.max())+1;new['ValidFrom']=change.EffectiveDate;new['ValidTo']='9999-12-31';new['RowIsCurrent']='Yes'
        attr='HQCity' if change.Attribute=='City' else change.Attribute;new[attr]=change.NewValue
        if attr=='HQCity':new['HQProvince']=provinces[change.NewValue]
        dim=pd.concat([dim,pd.DataFrame([new])],ignore_index=True)
    def customer_key(r):
        d=r.ShipDate.strftime('%Y-%m-%d'); candidates=dim.loc[dim.CustomerID.eq(r.CustomerID)&dim.ValidFrom.le(d)&dim.ValidTo.ge(d)]
        return int(candidates.iloc[0].CustomerKey) if len(candidates)==1 else -1
    good['CustomerKey']=good.apply(customer_key,axis=1)
    # S09 Add new lanes and Courier; retain all old keys.
    route=read('DimRoute(4).csv'); existing=set(zip(route.OriginCityKey,route.DestCityKey))
    for a,b in sorted(set(zip(good.OriginCityKey,good.DestCityKey))-existing):
        if a==-1 or b==-1:continue
        route.loc[len(route)]=[int(route.RouteKey.max())+1,a,b,f'{city_names[a]} -> {city_names[b]}','Not supplied','Not supplied']
    rmap={(r.OriginCityKey,r.DestCityKey):r.RouteKey for r in route.itertuples()}
    good['RouteKey']=[rmap.get((a,b),-1) for a,b in zip(good.OriginCityKey,good.DestCityKey)]
    mode=read('DimMode(4).csv');mode.loc[len(mode)]=[4,'Courier','ShaheenEx Courier Services','Up to 8 t','Parcel / subcontract road','No']
    good['ModeKey']=good.SourceMode.map({'Marine':1,'Rail':2,'Road':3,'Courier':4});good['DateKey']=good.ShipDate.dt.strftime('%Y%m%d').astype(int)
    feb=read('FactShipment_Feb.csv');march=read('FactShipment_March.csv');good['ShipmentKey']=np.arange(max(feb.ShipmentKey.max(),march.ShipmentKey.max())+1,max(feb.ShipmentKey.max(),march.ShipmentKey.max())+1+len(good))
    april=good[feb.columns].copy()
    for measure in ['WeightKG','FreightCostPKR']:
        assert april[measure].eq(april[measure].round()).all()
        april[measure]=april[measure].astype('int64')
    allfact=pd.concat([feb,march,april],ignore_index=True)
    # S10 Extend calendar without changing Q1 attributes.
    dates=read('DimDate(4).csv');newdates=[]
    for d in pd.date_range('2026-04-01','2026-04-30'):
        newdates.append([int(d.strftime('%Y%m%d')),d.strftime('%Y-%m-%d'),d.year,2,4,'April',d.day,(d.dayofweek+1)%7+1,d.day_name(),'Yes' if d.dayofweek>=5 else 'No','No'])
    dates=pd.concat([dates,pd.DataFrame(newdates,columns=dates.columns)],ignore_index=True)
    # S11 Publish detailed auditable tables. Staging remains an in-memory connection-only equivalent.
    outputs={'FactShipment_April':april,'RejectLog_April':reject,'DimCustomer':dim,'DimDate':dates,'DimRoute':route,'DimMode':mode,'DimCity':cities,'DimStatus':status,'FactShipment_FebToApril':allfact,'DuplicateLog_April':dups,'StructuralRemovalLog':pd.DataFrame(structural),'SourceProfile':profile}
    for name,table in outputs.items():table.to_csv(OUT/f'{name}.csv',index=False)
    return locals()

x = run_pipeline()
for name in ["staging","profile"]:
    print(name, len(x[name]))
display(x["profile"])
print("Failure sum:", int(x["failures"].sum().sum()))
print("Distinct failing rows:", int(x["failures"].any(axis=1).sum()))
assert x["failures"].sum().sum() == x["failures"].any(axis=1).sum() == 74

,Column,Blanks,DistinctRaw,RuleFailures,TextTransformOnly
0,ShipmentID,5,1771,5,1044
1,ShipDate,0,547,7,0
2,OriginCity,0,18,10,528
3,DestCity,0,21,9,188
4,WeightKG,2,102,19,0
5,FreightCostPKR,3,373,4,0
6,Status,0,30,9,991
7,CustomerID,2,50,11,109


staging 1805
profile 8
Failure sum: 74
Distinct failing rows: 74


**Profile conventions:** blanks exclude structural road rows; distinct raw counts exclude nulls and preserve raw case, whitespace, units and timestamps. Rule failures are evaluated using valid source-aware conversion and approved aliases; harmless formatting is not a failure. TextTransformOnly counts string conformance, including restoring rail IDs and canonicalizing customer aliases; date parsing and numeric unit conversion are separate typed operations.

**T2C.** Preserve courier timestamps in raw evidence and key the local shipping day to DateKey. A timestamp-to-date-key join would fail or invent many members for one day. ShipDate's 547 raw distinct values overstate dirtiness most: mixed date representations and 91 courier times are largely valid values at finer precision. The correct TOTAL footer is still an aggregate, not a shipment. Keeping it doubles a subtotal and violates the declared grain. The blank line and footer are recorded as structural removals, outside the 1,805-row staging population.

## T3 — Reject, route, remove

| Row class | Verdict | Reason / handling |
|---|---|---|
| Text/null weight | Reject | Cannot measure shipment; 10 |
| Negative weight | Reject | Impossible physical weight; 6 |
| Over mode capacity | Reject | Capacity contract exceeded; 3 |
| Blank ID | Reject | Cannot establish consignment identity; 5 |
| Invalid/non-April date | Reject | Invalid period/grain; 7 |
| Blank/text cost | Reject | Missing financial fact; 4 |
| Sound weight, negative cost | Load | Legitimate credit note; 3 |
| Unresolved city | Unknown city and route | Retain shipment; 18 loaded rows |
| Unresolved status | Unknown status | Retain shipment; 9 loaded rows |
| Unresolved customer | Unknown customer | Retain shipment; 11 loaded rows |
| Repeated normalized ID, same source | Remove duplicate | 55 repeated filings |
| Road/Courier same consignment | Remove courier duplicate | 15, retain Road record |
| Distinct road IDs sharing business attributes | Load | Nine genuine within-carrier pairs, all 18 retained (five Road, four Courier) |
| Blank line and TOTAL | Structural removal | 2, before staging |
| No fault | Load | Valid fact; counts below |

RejectLog has seven fields: **SourceFile** (text source), **SourceSHA256** (immutable input version), **Locator** (CSV line, worksheet row or JSONPath), **ShipmentID** (where supplied), **RawRecord** (complete serialized original record), **ReasonCode** (all failed validation reasons), **LoadBatch** (period). These enable locating, repairing and resubmitting evidence without inventing missing facts.

For JSON use `$.consignments[index]`, consignment ID where available, and source hash together. Array order can change, so the index alone is not stable across file versions. Rejecting prevents unmeasurable or unidentifiable rows from corrupting facts. Unknown preserves valid measures when classification is unresolved; rejecting those rows understates freight, while loading invalid measures as Unknown hides a substantive failure.

Deduplicate IDs within their source. For cross-carrier duplicates compare canonical day, origin, destination, weight and customer, and require Road versus Courier. Pair occurrences one-to-one, prefer Road, and never apply this business key within the same carrier.

In [3]:
display(x['reject'].ReasonCode.value_counts().rename_axis('Reason').to_frame('Rows'))
display(x['dups'].RemovalReason.value_counts().to_frame('Rows'))
display(x['outputs']['StructuralRemovalLog'])
good=x['good']; failures=x['failures']
print('Loaded Unknown rows:', int(((good.OriginCityKey==-1)|(good.DestCityKey==-1)|(good.StatusKey==-1)|(good.CustomerKey==-1)).sum()))
print('Loaded rows with none of these lookup faults:',int(((good.OriginCityKey!=-1)&(good.DestCityKey!=-1)&(good.StatusKey!=-1)&(good.CustomerKey!=-1)).sum()))
print('Raw failures assigned to rejected / loaded / removed:', int(failures.loc[x['reject'].index].any(axis=1).sum()), int(failures.loc[good.index].any(axis=1).sum()), int(failures.loc[x['dups'].index].any(axis=1).sum()))

,Rows
Reason,
WEIGHT_NOT_NUMERIC,10
INVALID_APRIL_DATE,7
WEIGHT_NONPOSITIVE,6
MISSING_SHIPMENT_ID,5
COST_NOT_NUMERIC,4
WEIGHT_OVER_CAPACITY,3


,Rows
RemovalReason,
Same source / ShipmentID,55
Road / Courier same consignment,15


,SourceFile,Locator,Reason
0,road_april.csv,line 741,Blank line
1,road_april.csv,line 742,TOTAL footer


Loaded Unknown rows: 38
Loaded rows with none of these lookup faults: 1662
Raw failures assigned to rejected / loaded / removed: 35 38 1


## T4 — Reconciliation ladder and wrong pipelines

In [4]:
staging=x['staging']; ladder=[]
for source,g in staging.groupby('SourceFile',sort=False):
    ladder.append([source,len(g),pd.to_numeric(g.WeightKG,errors='coerce').sum(),pd.to_numeric(g.FreightCostPKR,errors='coerce').sum()])
ladder += [['Staging',len(staging),None,None],['Rejected',len(x['reject']),None,None],['Same-ID duplicates',55,None,None],['Cross-carrier duplicates',15,None,None],['April loaded',len(x['april']),x['april'].WeightKG.sum(),x['april'].FreightCostPKR.sum()]]
ladder=pd.DataFrame(ladder,columns=['Stage','Rows','RawWeightSum_or_LoadedKG','CostPKR'])
display(ladder);ladder.to_csv(OUT/'ReconciliationLadder.csv',index=False)
print('Identity residual:',len(staging)-len(x['april'])-len(x['reject'])-len(x['dups']))
# Reproduce the supplied wrong answer: sort by ID, then deduplicate the business key globally.
ordered=good.sort_values('ShipmentID')
wrong=ordered.loc[ordered.duplicated(x['bk'],keep='first')]
display(wrong[['ShipmentID']+x['bk']+['FreightCostPKR']])
print('Real shipments lost:',len(wrong),'kg:',wrong.WeightKG.sum(),'PKR:',wrong.FreightCostPKR.sum())
assert len(wrong)==9 and wrong.WeightKG.sum()==33550 and wrong.FreightCostPKR.sum()==313000

,Stage,Rows,RawWeightSum_or_LoadedKG,CostPKR
0,marine_april.csv,256,4445000.0,36783500.0
1,rail_april.xlsx,318,9305.5,86026500.0
2,road_april.csv,739,4660200.0,34970000.0
3,courier_april.json,492,390250000.0,9776000.0
4,Staging,1805,NaN,NaN
5,Rejected,35,NaN,NaN
6,Same-ID duplicates,55,NaN,NaN
7,Cross-carrier duplicates,15,NaN,NaN
8,April loaded,1700,17915700.0,158037000.0


,ShipmentID,ShipDate,OriginCityKey,DestCityKey,WeightKG,CustomerID,FreightCostPKR
1259,RD-6237,2026-04-25,3,5,6400.0,CU-134,56000.0
1225,RD-6377,2026-04-13,5,10,6500.0,CU-103,53000.0
1299,RD-6413,2026-04-12,4,3,6300.0,CU-138,49000.0
968,RD-6604,2026-04-22,1,11,6500.0,CU-109,44500.0
624,RD-6650,2026-04-19,7,4,5500.0,CU-108,39500.0
1545,SX-10349,2026-04-09,4,3,300.0,CU-111,7500.0
1673,SX-10359,2026-04-30,3,7,800.0,CU-107,28500.0
1602,SX-10363,2026-04-05,4,3,350.0,CU-126,11500.0
1631,SX-10375,2026-04-11,1,11,900.0,CU-134,23500.0


Identity residual: 0
Real shipments lost: 9 kg: 33550.0 PKR: 313000.0


The four raw sums mix kilograms, tonnes and grams and must not be added. Numeric coercion also skips ten bad weights; a successful sum is not validation.

| Wrong result | Failure | T8 detection |
|---|---|---|
| 1,710; correct kg/cost | Ten text-weight rows retained as null/zero measures; totals skip them | Reject count #2 and loaded rows #4; identity #1–4 |
| 1,691; 17,882,150 kg; 157,724,000 PKR | Global business-key dedup deletes nine genuine shipments (five Road, four Courier) | Duplicates #3, rows #4 and totals #5–6; two IDs shown in table above |
| 1,682; 17,734,700 kg | Deletes eighteen unresolved-city shipments instead of routing | Rows #4, kg #5 and Unknown city #7 |
| 1,700; 8,979,894.75 kg | Rail tonnes not multiplied by 1,000 | Weight #5; blended cost/kg becomes about 17.60 |


## T5 — Slowly changing dimensions

| Type | Behavior | Rows after changes | As-of province available? | March Punjab kg | Use |
|---|---|---:|---|---:|---|
| 0 | Retain originals | 41 | March only; later moves unrepresented | 7,566,850 | Immutable signup date |
| 1 | Overwrite attributes | 41 | No | 7,344,400 | Corrected/current descriptive name |
| 2 | New surrogate per effective version | 47 | Yes | 7,566,850 | Historical location/sector attribution |
| 3 | Previous/current attribute columns | 41 | Limited; requires effective dates and conditional query | 7,344,400 if joined to current province | Current versus previous comparison |

**Per-change policy:** Type 2 for CU-105, CU-120 and CU-116 HQ moves; preserve original period attribution. Type 2 for CU-123 sector reclassification where historical sector analysis matters. CU-132 can use Type 3 for current versus previous sector comparisons, with the limitation that repeated changes lose deeper history; use Type 2 for unrestricted as-of sector reporting. CU-140 is the same business and premises: recommend Type 1 for its display name; no weight shifts province, though historical labels change. SignedUp is Type 0, except a separately approved source error correction.

**Implementation versus recommendation:** T5D explicitly requires *all six changes* as Type 2 and 47 rows, so the technical lab applies six versions. This meets that specified build while the policy answers distinguish a rename from an analytically meaningful history change. Validity is inclusive, with the old row ending the day before the change.

March facts retain old surrogate keys because those keys identify the customer version at shipment time. Repointing facts to the current key rewrites published province/sector history without changing total freight. CU-116 moves into Punjab, offsetting CU-105 and CU-120 moving out. Gross reassigned freight measures the actual history affected; net province differences hide offsetting movements.

In [5]:
dim=x['dim']; march=x['march']; changes=x['changes']; base=x['base']
published=march.merge(base[['CustomerKey','CustomerID','HQProvince']],on='CustomerKey',validate='many_to_one')
overwrite=base.copy()
for ch in changes.itertuples():
    if ch.Attribute=='City':overwrite.loc[overwrite.CustomerID.eq(ch.CustomerID),['HQCity','HQProvince']]=[ch.NewValue,x['provinces'][ch.NewValue]]
after=march.merge(overwrite[['CustomerKey','HQProvince']],on='CustomerKey',validate='many_to_one')
board=pd.concat([published.groupby('HQProvince').WeightKG.sum().rename('AsPublished'),after.groupby('HQProvince').WeightKG.sum().rename('AfterType1')],axis=1)
board['Difference']=board.AfterType1-board.AsPublished;display(board)
print('Gross moved kg:', published.loc[published.CustomerID.isin(['CU-105','CU-120','CU-116']),'WeightKG'].sum())
print('Net positive movement:',board.Difference.clip(lower=0).sum())
splits=[]
for ch in changes[changes.Attribute.eq('City')].itertuples():
    g=good.loc[good.CustomerID.eq(ch.CustomerID)];before=g.ShipDate.lt(ch.EffectiveDate)
    versions=dim.loc[dim.CustomerID.eq(ch.CustomerID)].sort_values('CustomerKey')
    splits.append([ch.CustomerID,ch.EffectiveDate,int(before.sum()),int((~before).sum()),g.loc[before,'WeightKG'].sum(),g.loc[~before,'WeightKG'].sum(),*versions.CustomerKey.tolist()])
splits=pd.DataFrame(splits,columns=['CustomerID','EffectiveDate','BeforeRows','AfterRows','OldProvinceKG','NewProvinceKG','OldKey','NewKey']);display(splits)
display(pd.Series({'All versions':len(dim),'Current':dim.RowIsCurrent.eq('Yes').sum(),'Expired':dim.RowIsCurrent.eq('No').sum(),'Businesses':dim.CustomerID.nunique(),'Open ended':dim.ValidTo.eq('9999-12-31').sum()}).to_frame('Count'))
board.to_csv(OUT/'MarchProvince_Type1Comparison.csv');splits.to_csv(OUT/'AprilCustomerSplits.csv',index=False)

,AsPublished,AfterType1,Difference
HQProvince,,,
Balochistan,1863700,1863700,0
Islamabad ICT,340650,609150,268500
Khyber Pakhtunkhwa,2162900,1832450,-330450
Punjab,7566850,7344400,-222450
Sindh,5607950,5892350,284400


,CustomerID,EffectiveDate,BeforeRows,AfterRows,OldProvinceKG,NewProvinceKG,OldKey,NewKey
0,CU-105,2026-04-07,6,44,32350.0,528000.0,5,41
1,CU-120,2026-04-12,10,35,89350.0,369050.0,20,42
2,CU-116,2026-04-19,21,16,232000.0,217100.0,16,43


,Count
All versions,47
Current,41
Expired,6
Businesses,40
Open ended,41


Gross moved kg: 883350
Net positive movement: 552900


## T6 — OLAP navigation

In [6]:
fact=x['allfact']; city=x['cities']; mode=x['mode']
cube=fact.merge(city[['CityKey','CityName','Province','Region']],left_on='DestCityKey',right_on='CityKey',validate='many_to_one').merge(mode[['ModeKey','ModeName']],on='ModeKey',validate='many_to_one')
cube['Month']=cube.DateKey.astype(str).str[:6]
down=cube.loc[cube.Month.eq('202604')&cube.Province.eq('Punjab')].groupby('CityName').WeightKG.sum().sort_values(ascending=False)
slice_=cube.loc[cube.ModeName.eq('Courier')]
dice=cube.loc[cube.Month.eq('202604')&cube.ModeName.isin(['Marine','Rail'])&cube.Province.isin(['Sindh','Punjab'])]
olap=pd.DataFrame({'Answer':[len(fact),fact.WeightKG.sum(),fact.FreightCostPKR.sum(),cube.loc[cube.Region.eq('North'),'WeightKG'].sum(),cube.loc[cube.Province.eq('Punjab'),'WeightKG'].sum(),down.iloc[0],len(slice_),slice_.WeightKG.sum(),len(dice),dice.WeightKG.sum()]},index=['Integrated rows','Integrated kg','Integrated PKR','North kg','Punjab kg','Largest April Punjab city kg','Courier rows','Courier kg','Dice rows','Dice kg'])
display(olap);display(down)
expected=[2934,36036400,309882000,21089950,22469700,3349600,460,281050,414,10019750]
assert olap.Answer.tolist()==expected
weight=cube.pivot_table(index='ModeName',columns='Month',values='WeightKG',aggfunc='sum',margins=True,margins_name='All')
cost=cube.pivot_table(index='ModeName',columns='Month',values='FreightCostPKR',aggfunc='sum',margins=True,margins_name='All')
cpk=cost/weight;display(cpk.round(4));cpk.to_csv(OUT/'CostPerKG_ModeMonth.csv')
without=cube.loc[cube.Month.eq('202604')&cube.ModeName.ne('Courier')];print('April excluding Courier:',without.FreightCostPKR.sum()/without.WeightKG.sum())
agg=cube.groupby(['Province','Month'],as_index=False)[['WeightKG','FreightCostPKR']].sum()
print('Destination province/month aggregate rows:',len(agg));display(agg)
agg.to_csv(OUT/'DestinationProvinceMonthAggregate.csv',index=False)
hq=fact.merge(dim[['CustomerKey','HQProvince']],on='CustomerKey',validate='many_to_one');hq['Month']=hq.DateKey.astype(str).str[:6]
hq_agg=hq.pivot_table(index='HQProvince',columns='Month',values='WeightKG',aggfunc='sum',fill_value=0);display(hq_agg);hq_agg.to_csv(OUT/'Board_HQProvinceMonth.csv')

,Answer
Integrated rows,2934
Integrated kg,36036400
Integrated PKR,309882000
North kg,21089950
Punjab kg,22469700
Largest April Punjab city kg,3349600
Courier rows,460
Courier kg,281050
Dice rows,414
Dice kg,10019750


CityName
Lahore        3349600
Multan        2111900
Faisalabad    1886800
Sialkot       1810750
Rawalpindi    1486750
Gujranwala     372650
Name: WeightKG, dtype: int64

Month,202602,202603,202604,All
ModeName,,,,
Courier,NaN,NaN,31.2151,31.2151
Marine,7.8354,7.5337,8.1815,7.8531
Rail,8.7553,9.4187,9.1036,9.2529
Road,7.8674,7.0659,7.4482,7.2668
All,8.3090,8.3820,8.8211,8.5991


,Province,Month,WeightKG,FreightCostPKR
0,Balochistan,202602,20000,188000
1,Balochistan,202603,476500,3379500
2,Islamabad ICT,202602,22000,197000
3,Islamabad ICT,202603,824700,5905000
4,Islamabad ICT,202604,2075450,17857500
5,Khyber Pakhtunkhwa,202602,61900,623000
6,Khyber Pakhtunkhwa,202603,2186950,18088000
7,Khyber Pakhtunkhwa,202604,2175300,19764500
8,Punjab,202602,388800,3247000
9,Punjab,202603,11062450,95340000


Month,202602,202603,202604
HQProvince,,,
Balochistan,0,1863700,1669300
Islamabad ICT,0,340650,628100
Khyber Pakhtunkhwa,0,2162900,2366750
Punjab,0,7566850,7198200
Sindh,0,5607950,5931450
Unknown,578650,0,121900


April excluding Courier: 8.464245108352022
Destination province/month aggregate rows: 15


Absent Courier cells in February/March are **not applicable**, not zero cost/kg: no shipments exist for that mode in those periods. All margins use SUM(cost)/SUM(weight), never averages of cell ratios.

**Board sentence:** “Blended cost/kg rose from 8.31 to 8.38 to 8.82, partly because April introduces a higher-cost, low-weight Courier mix. April excluding Courier is 8.46; the blend alone does not establish a price rise. Compare like-for-like modes, lanes and services before attributing the change to prices.”

The destination province × month aggregate above summarizes 2,934 facts. Rebuild affected partitions on each load, correction or dimension change. New Type 2 versions leave correctly keyed historical as-of aggregates intact; rebuild April and any genuinely corrected effective periods. If the cache instead groups *all historical shipments by current customer province*, rebuild every month in which that customer shipped. This distinction resolves the brief's broad invalidation statement: current-view caches invalidate across history, while preserved as-of history must not be reassigned. Include Unknown province when it occurs; it is a real reporting bucket.

## T7 — Selectivity, index and aggregate decisions

In [7]:
cols=['DateKey','OriginCityKey','DestCityKey','RouteKey','ModeKey','StatusKey','CustomerKey','ShipmentID']
usage=['every page','sometimes','sometimes','sometimes','every page','sometimes','sometimes','never']
selectivity=pd.DataFrame({'Column':cols,'Distinct':[fact[c].nunique() for c in cols],'Rows':len(fact),'DashboardFilter':usage})
selectivity['Selectivity']=selectivity.Distinct/selectivity.Rows;display(selectivity)
selectivity.to_csv(OUT/'IndexSelectivity.csv',index=False)
display(pd.DataFrame([['Star',2,1,len(fact)],['Snowflake',4,3,len(fact)*3],['Aggregate direct',1,0,0]],columns=['Path','Tables','Joins','JoinProbes']))
print('Aggregate direct scan:',len(agg),'rows; zero joins, but not zero reads.')

,Column,Distinct,Rows,DashboardFilter,Selectivity
0,DateKey,73,2934,every page,0.024881
1,OriginCityKey,12,2934,sometimes,0.004090
2,DestCityKey,14,2934,sometimes,0.004772
3,RouteKey,25,2934,sometimes,0.008521
4,ModeKey,4,2934,every page,0.001363
5,StatusKey,4,2934,sometimes,0.001363
6,CustomerKey,47,2934,sometimes,0.016019
7,ShipmentID,2934,2934,never,1.000000


,Path,Tables,Joins,JoinProbes
0,Star,2,1,2934
1,Snowflake,4,3,8802
2,Aggregate direct,1,0,0


Aggregate direct scan: 15 rows; zero joins, but not zero reads.


Choose a **DateKey index** for a relational row-store backing this model: period predicates prune relevant facts and DateKey participates in most analytic queries with better filtering power than the four ModeKey values. ShipmentID is maximally selective but unused by the dashboard. The filter-frequency column expresses the assumed dashboard design, since no actual report file was supplied. Inspect execution plans before deploying. Power BI import uses columnar storage, so this recommendation concerns the warehouse database, not creating a B-tree inside a PBIX.

**Write cost:** 1,700 April rows × one new index = **1,700 index-entry insertions**, plus engine-specific page splits and logging. Maintenance of existing indexes is additional.

An index cannot fix (1) expensive visual/DAX queries or remote/rendering overhead, or (2) an unnecessarily long snowflake join path / repeated detail aggregation. This supplied DimCity already embeds Province and Region, so the lab's hypothetical snowflake is avoidable; it is not present in the delivered star. The actual report's slow query cannot be diagnosed without a PBIX and timing evidence. A repeated province/month aggregation is demonstrable, but 2,934 rows alone do not establish a bottleneck. Pre-aggregation may reduce repeated work but adds refresh and correctness maintenance. Measure first before persisting another production cache. The CSV aggregate here demonstrates the arithmetic.

The prescribed join-probe metric is 2,934 versus 8,802 versus 0; the aggregate still scans its own rows. A zero-join path is not a zero-cost query.

## T8 — Definition of done and unresolved values

In [8]:
april=x['april']; unknown_city=(april.OriginCityKey.eq(-1)|april.DestCityKey.eq(-1))
punjab_march=hq.loc[hq.Month.eq('202603')&hq.HQProvince.eq('Punjab'),'WeightKG'].sum()
actual=[len(staging),len(x['reject']),len(x['dups']),len(april),april.WeightKG.sum(),april.FreightCostPKR.sum(),int(unknown_city.sum()),int(april.StatusKey.eq(-1).sum()),int(april.CustomerKey.eq(-1).sum()),len(dim),len(x['dates']),punjab_march]
expected=[1805,35,70,1700,17915700,158037000,18,9,11,47,120,7566850]
checks=pd.DataFrame({'Check':['Staging rows','Rejected rows','Duplicates removed','April rows','April kg','April PKR','Unknown city rows','Unknown status rows','Unknown customer rows','Customer versions','Calendar rows','March HQ Punjab kg'],'Actual':actual,'Expected':expected})
checks['Status']=np.where(checks.Actual.eq(checks.Expected),'PASS','FIX');display(checks)
assert checks.Status.eq('PASS').all()
assert len(staging)==len(april)+len(x['reject'])+len(x['dups'])
assert len(fact)==2934 and fact.WeightKG.sum()==36036400 and fact.FreightCostPKR.sum()==309882000
assert april.FreightCostPKR.lt(0).sum()==3
assert fact.CustomerKey.eq(-1).sum()==45
assert len(x['route'])==25 and len(x['mode'])==5 and april.RouteKey.eq(-1).sum()==18
assert dim.RowIsCurrent.eq('Yes').sum()==41 and dim.CustomerID.nunique()==40
pd.testing.assert_frame_equal(fact.iloc[:34].reset_index(drop=True),x['feb'])
pd.testing.assert_frame_equal(fact.iloc[34:1234].reset_index(drop=True),x['march'])
assert len(pd.date_range(x['dates'].FullDate.min(),x['dates'].FullDate.max()))==len(x['dates'])==120
# Every loaded customer key must be valid on the fact's date, except the Unknown member.
for r in good.loc[good.CustomerKey.ne(-1)].itertuples():
    v=dim.loc[dim.CustomerKey.eq(r.CustomerKey)].iloc[0];d=r.ShipDate.strftime('%Y-%m-%d');assert v.ValidFrom<=d<=v.ValidTo
# Primary keys and referential integrity.
assert fact.ShipmentKey.is_unique and fact.ShipmentID.is_unique
for key,dimension,dimkey in [('DateKey',x['dates'],'DateKey'),('OriginCityKey',city,'CityKey'),('DestCityKey',city,'CityKey'),('RouteKey',x['route'],'RouteKey'),('ModeKey',mode,'ModeKey'),('StatusKey',x['status'],'StatusKey'),('CustomerKey',dim,'CustomerKey')]:
    assert dimension[dimkey].is_unique
    assert set(fact[key]) <= set(dimension[dimkey])
checks.to_csv(OUT/'ValidationChecks.csv',index=False)
print('12 / 12 checks PASS; carryforward facts unchanged; keys and temporal lookup valid.')

,Check,Actual,Expected,Status
0,Staging rows,1805,1805,PASS
1,Rejected rows,35,35,PASS
2,Duplicates removed,70,70,PASS
3,April rows,1700,1700,PASS
4,April kg,17915700,17915700,PASS
5,April PKR,158037000,158037000,PASS
6,Unknown city rows,18,18,PASS
7,Unknown status rows,9,9,PASS
8,Unknown customer rows,11,11,PASS
9,Customer versions,47,47,PASS


12 / 12 checks PASS; carryforward facts unchanged; keys and temporal lookup valid.


In [9]:
raw=staging.loc[good.index]
unknowns=[]
city_values=[]
for field,key in [('OriginCity','OriginCityKey'),('DestCity','DestCityKey')]:city_values += raw.loc[good[key].eq(-1),field].astype(str).str.strip().tolist()
status_values=raw.loc[good.StatusKey.eq(-1),'Status'].astype(str).str.strip().unique().tolist()
customer_values=raw.loc[good.CustomerKey.eq(-1),'CustomerID'].fillna('(blank)').astype(str).str.strip().unique().tolist()
unknowns=[['City',sorted(set(city_values)),int(unknown_city.sum()),'Ask operations to approve canonical aliases or add verified cities; never guess.'],['Status',sorted(status_values),int(april.StatusKey.eq(-1).sum()),'Ask carrier/business owners to approve status semantics and terminal/exception flags, including codes 7 and 9.'],['Customer',sorted(customer_values),int(april.CustomerKey.eq(-1).sum()),'Ask CRM owner to register or match references and repair blank customer IDs.']]
unresolved=pd.DataFrame(unknowns,columns=['Group','RawValues','LoadedRows','NextMonthProposal']);display(unresolved)
unresolved.to_csv(OUT/'UnresolvedValues.csv',index=False)
print('All-quarter Unknown customer rows:',int(fact.CustomerKey.eq(-1).sum()),'= 34 February without customer data + 11 April unresolved.')
# S12 Replay the same batch: no accumulation of fact rows or dimension versions.
y=run_pipeline()
for name in ['FactShipment_April','DimCustomer','FactShipment_FebToApril','RejectLog_April']:
    pd.testing.assert_frame_equal(x['outputs'][name],y['outputs'][name])
print('Second refresh is identical: 1,700 April facts, 47 customer versions, 2,934 integrated facts.')

,Group,RawValues,LoadedRows,NextMonthProposal
0,City,"[Bin Qasim Terminal, Chaman Border, Hub Chowki...",18,Ask operations to approve canonical aliases or...
1,Status,"[7, 9, HELD AT CUSTOMS, Short Landed, rto]",9,Ask carrier/business owners to approve status ...
2,Customer,"[(blank), CU-951, CU-952, CU-953, CU-954]",11,Ask CRM owner to register or match references ...


All-quarter Unknown customer rows: 45 = 34 February without customer data + 11 April unresolved.
Second refresh is identical: 1,700 April facts, 47 customer versions, 2,934 integrated facts.


## One-page reflection

I would defend Type 2 most strongly for the three HQ moves, since the board groups shipments by customer province. CU-116 moved into Punjab while CU-105 and CU-120 moved out. Overwriting customer locations shifts 883,350 kg of March history even though both province tables retain the correct 17,542,050 kg total. The sum of positive province differences is only 552,900 kg because moves offset each other. Islamabad ICT's increase would be especially noticeable, while Punjab's smaller percentage decline could escape review.

I would keep signup dates as Type 0, track sector history with Type 2 when past sector analysis matters, and consider Type 3 for a limited previous/current sector comparison. The CU-140 rename is a Type 1 display-name change in my policy recommendation. The required all-Type-2 lab is implemented separately to satisfy its explicit 47-row specification.

I retained 18 unresolved-city rows, nine unresolved-status rows and eleven unresolved-customer rows. Their exact six city spellings, five statuses and five customer references appear above and in UnresolvedValues.csv. These rows represent real measured business, so Unknown preserves their freight while business owners clarify classification. Deleting them conceals data quality instead of resolving it. The total 45 Unknown-customer facts includes February's 34 rows, which never contained customer information.

For the board, I would explain the change in blend before describing any price trend: Courier's arrival raises April's blended cost/kg to 8.82, while excluding Courier gives 8.46. Neither number alone establishes a like-for-like rate increase.

I would investigate DateKey indexing in a relational warehouse, costing 1,700 additional index entries for this April load. I would examine actual query plans and report timings before materializing a production aggregate at this small scale. Preserving published facts matters more than making a wrong answer return faster.

**Reproducibility:** all outputs are derived from the supplied files. Input hashes and exact locators support rejects; duplicate and structural-removal logs account for other removals. A repeated run replaces the deterministic April snapshot rather than appending a second copy. Holiday values for newly generated April dates default to No because no holiday calendar was supplied; this does not affect any requested metric. New lanes retain their keys and names; corridor/distance descriptors are marked Not supplied rather than inferred.